# <p align = "center"> Looking at Projects and Sessions Statistics </p>

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, Path("../../..") )

In [14]:
from xaidar.filesUtils import loadPickle, savePyObj

xchemRawKeysDir = Path("../../../data/s3Sizes/xchem/raw")

lookExample = True
rootDirNames = ["data", "dataset"]
saveDir = Path("./xchemSeshs")
saveDir.mkdir( parents = True, exist_ok = True )

for fragPath in xchemRawKeysDir.iterdir(): 
    rootDirContent = { dir : {"year": [], "session": [], "size": [] } for dir in rootDirNames}
    if fragPath.is_file():
        fragName = fragPath.name[:-10]
        print( f"Loading Fragment: {fragName}")
        frag = loadPickle( fragPath )
        for key, value in frag.items():
            objPath = key.split("/")
            rootDir = objPath[0]
            
            if rootDir in rootDirNames:
                rootDirContent[rootDir]["year"].append( objPath[1])
                rootDirContent[rootDir]["session"].append( objPath[2])
                if type( value) == int:
                    rootDirContent[rootDir]["size"].append( value )
                else:
                    rootDirContent[rootDir]["size"].append( None)
            else:
                print( f"New root Dir in xchem: {rootDir}")
    savePath = saveDir / f"xc_{fragName}_rootContent.pkl"
    savePyObj( rootDirContent, savePath )


print( "Done")

Loading Fragment: frag10
Loading Fragment: frag11
Loading Fragment: frag12
Loading Fragment: frag13
Loading Fragment: frag14
Loading Fragment: frag1
Loading Fragment: frag2
Loading Fragment: frag3
Loading Fragment: frag4
Loading Fragment: frag5
Loading Fragment: frag6
Loading Fragment: frag7
Loading Fragment: frag8
Loading Fragment: frag9
Done


### xchem:  data/

In [ ]:
# xchem : data total projects and sessions
import re
from xaidar.filesUtils import loadPickle

st_sessions = set()
st_projects = set()

xchemSeshDir = Path("./xchemSeshs")
for fragFilePath in xchemSeshDir.iterdir():
    if fragFilePath.is_file():
        # print( f"Loading {fragFilePath.name}")
        rootDirContent = loadPickle( fragFilePath)

        lst_sessions = rootDirContent["data"]["session"]
        st_sessions.update( lst_sessions )

        lst_projects = [ session[:re.search( "-[0-9]+$", session).start() ] for session in lst_sessions]
        st_projects.update( lst_projects)


print("\nFor xchem : data/ there are:")
print( f"\tNumber of Projects: {len(st_projects )}")
print( f"\tNumber of Sessions: {len(st_sessions) }")



Loading xc_frag10_rootContent.pkl
Loading xc_frag11_rootContent.pkl
Loading xc_frag12_rootContent.pkl
Loading xc_frag13_rootContent.pkl
Loading xc_frag14_rootContent.pkl
Loading xc_frag1_rootContent.pkl
Loading xc_frag2_rootContent.pkl
Loading xc_frag3_rootContent.pkl
Loading xc_frag4_rootContent.pkl
Loading xc_frag5_rootContent.pkl
Loading xc_frag6_rootContent.pkl
Loading xc_frag7_rootContent.pkl
Loading xc_frag8_rootContent.pkl
Loading xc_frag9_rootContent.pkl
For xchem : data/ there are:
	Number of Projects: 154
	Number of Sessions: 739


- Per Session

In [27]:
# xchem : data/ Per session Data
import numpy as np
from xaidar.filesUtils import roundBytes

dataSessionsContent = { session : {"Storage Size": 0, "Number of Files": 0} for session in st_sessions }


xchemSeshDir = Path("./xchemSeshs")
for fragFilePath in xchemSeshDir.iterdir():
    if fragFilePath.is_file():
        print( f"Loading {fragFilePath.name}")
        rootDirContent = loadPickle(fragFilePath)
        lst_sessions = rootDirContent["data"]["session"] 
        lst_sizes = rootDirContent["data"]["size"] 
        del rootDirContent
        for session, size in zip( lst_sessions, lst_sizes ):
            if type(size) == int: dataSessionsContent[session]["Storage Size"] += size
            dataSessionsContent[session]["Number of Files"] += 1

seshSizes = [ dataSessionsContent[sesh]["Storage Size"] for sesh in dataSessionsContent.keys() ]
seshfileCount = [ dataSessionsContent[sesh]["Number of Files"] for sesh in dataSessionsContent.keys() ]

print( "Done Loading")        

meanSeshSize = roundBytes( np.mean( seshSizes ) )
medianSeshSize = roundBytes( np.median(seshSizes ) )
interQRange = roundBytes( np.percentile( seshSizes, 75) - np.percentile( seshSizes, 25) )

meanSeshCount = np.mean( seshfileCount )
medianSeshCount = np.median( seshfileCount )
interQRangeCount = np.percentile( seshfileCount, 75) - np.percentile( seshfileCount, 25)

print( "For xchem : data/ per session, we have:")
print( f"\tMean Session Size: {meanSeshSize }")
print( f"\tMedian Session Size: {medianSeshSize }")
print( f"\tInter Quartile Range of Session Size: {interQRange }")
print( f"\tMean Number of Files per Session: {meanSeshCount:.2f}")
print( f"\tMedian Number of Files per Session: {medianSeshCount:.2f}")
print( f"\tInter Quartile Range of Number of Files per Session: {interQRangeCount:.2f}")
print( "Done")

Loading xc_frag10_rootContent.pkl
Loading xc_frag11_rootContent.pkl
Loading xc_frag12_rootContent.pkl
Loading xc_frag13_rootContent.pkl
Loading xc_frag14_rootContent.pkl
Loading xc_frag1_rootContent.pkl
Loading xc_frag2_rootContent.pkl
Loading xc_frag3_rootContent.pkl
Loading xc_frag4_rootContent.pkl
Loading xc_frag5_rootContent.pkl
Loading xc_frag6_rootContent.pkl
Loading xc_frag7_rootContent.pkl
Loading xc_frag8_rootContent.pkl
Loading xc_frag9_rootContent.pkl
Done Loading
For xchem : data/ per session, we have:
	Mean Session Size: (np.float64(210.7), 'GB')
	Median Session Size: (np.float64(279.74), 'KB')
	Inter Quartile Range of Session Size: (np.float64(54.09), 'GB')
	Mean Number of Files per Session: 174970.57
	Median Number of Files per Session: 8.00
	Inter Quartile Range of Number of Files per Session: 44160.50
Done


- Per Project

In [28]:
# xchem : data/ Per Project Data

import numpy as np
from xaidar.filesUtils import roundBytes

dataProjectsContent = { project : {"Storage Size": 0, "Number of Files": 0} for project in st_projects }


xchemSeshDir = Path("./xchemSeshs")
for fragFilePath in xchemSeshDir.iterdir():
    if fragFilePath.is_file():
        print( f"Loading {fragFilePath.name}")
        rootDirContent = loadPickle(fragFilePath)
        lst_sessions = rootDirContent["data"]["session"] 
        lst_sizes = rootDirContent["data"]["size"] 
        del rootDirContent
        for session, size in zip( lst_sessions, lst_sizes ):
            project = session[:re.search( "-[0-9]+$", session).start() ]
            if type(size) == int: dataProjectsContent[project]["Storage Size"] += size
            dataProjectsContent[project]["Number of Files"] += 1

projSizes = [ dataProjectsContent[proj]["Storage Size"] for proj in dataProjectsContent.keys() ]
projfileCount = [ dataProjectsContent[proj]["Number of Files"] for proj in dataProjectsContent.keys() ]

print( "Done Loading")        

meanProjhSize = roundBytes( np.mean( projSizes ) )
medianProjSize = roundBytes( np.median(projSizes ) )
projinterQRange = roundBytes( np.percentile( projSizes, 75) - np.percentile( projSizes, 25) )

meanProjCount = np.mean( projfileCount )
medianProjCount = np.median( projfileCount )
projinterQRangeCount = np.percentile( projfileCount, 75) - np.percentile( projfileCount, 25)

print( "For xchem : data/ per session, we have:")
print( f"\tMean Project Size: {meanProjhSize }")
print( f"\tMedian Project Size: {medianProjSize }")
print( f"\tInter Quartile Range of Project Size: {projinterQRange }")
print( f"\tMean Number of Files per Project: {meanProjCount:.2f}")
print( f"\tMedian Number of Files per Project: {medianProjCount:.2f}")
print( f"\tInter Quartile Range of Number of Files per Project: {projinterQRangeCount:.2f}")
print( "Done")


Loading xc_frag10_rootContent.pkl
Loading xc_frag11_rootContent.pkl
Loading xc_frag12_rootContent.pkl
Loading xc_frag13_rootContent.pkl
Loading xc_frag14_rootContent.pkl
Loading xc_frag1_rootContent.pkl
Loading xc_frag2_rootContent.pkl
Loading xc_frag3_rootContent.pkl
Loading xc_frag4_rootContent.pkl
Loading xc_frag5_rootContent.pkl
Loading xc_frag6_rootContent.pkl
Loading xc_frag7_rootContent.pkl
Loading xc_frag8_rootContent.pkl
Loading xc_frag9_rootContent.pkl
Done Loading
For xchem : data/ per session, we have:
	Mean Session Size: (np.float64(1.01), 'TB')
	Median Session Size: (np.float64(76.33), 'GB')
	Inter Quartile Range of Session Size: (np.float64(329.66), 'GB')
	Mean Number of Files per Session: 839631.52
	Median Number of Files per Session: 58932.00
	Inter Quartile Range of Number of Files per Session: 203141.25
Done


Per Year

In [37]:
from xaidar.filesUtils import loadPickle, savePyObj

xchemRawKeysDir = Path("../../../data/s3Sizes/xchem/raw")

lookExample = True
rootDirNames = ["data"]
saveDir = Path("./xchemYears")
saveDir.mkdir( parents = True, exist_ok = True )

knownYears = ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022']

for fragPath in xchemRawKeysDir.iterdir(): 
    rootDirContent = { dir : {"year": [], "session": [], "size": [] } for dir in rootDirNames}
    if fragPath.is_file():
        fragName = fragPath.name[:-10]
        print( f"Loading Fragment: {fragName}")
        frag = loadPickle( fragPath )
        for key, value in frag.items():
            objPath = key.split("/")
            rootDir = objPath[0]
            
            if rootDir in rootDirNames:
                if objPath[1] in knownYears: rootDirContent[rootDir]["year"].append( objPath[1])
                else:rootDirContent[rootDir]["year"].append( "Unknown")
                rootDirContent[rootDir]["session"].append( objPath[2])
                if type( value) == int:
                    rootDirContent[rootDir]["size"].append( value )
                else:
                    rootDirContent[rootDir]["size"].append( None)
            elif  rootDir == "dataset":
                pass
            else:
                print( f"New root Dir in xchem: {rootDir}")
                
    savePath = saveDir / f"xc_{fragName}_yearContent.pkl"
    savePyObj( rootDirContent, savePath )


print( "Done")

Loading Fragment: frag10
Loading Fragment: frag11
Loading Fragment: frag12
Loading Fragment: frag13
Loading Fragment: frag14
Loading Fragment: frag1
Loading Fragment: frag2
Loading Fragment: frag3
Loading Fragment: frag4
Loading Fragment: frag5
Loading Fragment: frag6
Loading Fragment: frag7
Loading Fragment: frag8
Loading Fragment: frag9
Done


In [21]:
# # xchem : data/ Per Year Data

import numpy as np
from xaidar.filesUtils import roundBytes



knownYears = ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022']
st_years = set( knownYears + ["Unknown"] )

dataYearsContent = { year : {"Storage Size": 0, "Number of Files": 0, "Sessions": set(), "Projects": set() } for year in st_years }


xchemYearsDir = Path("./xchemYears")
for fragFilePath in xchemYearsDir.iterdir():
    if fragFilePath.is_file():
        print( f"Loading {fragFilePath.name}")
        rootDirContent = loadPickle(fragFilePath)

        lst_years = rootDirContent["data"]["year"] 
        lst_sizes = rootDirContent["data"]["size"] 
        lst_sessions = rootDirContent["data"]["session"]
        lst_projects = [ session[:re.search( "-[0-9]+$", session).start() ] for session in lst_sessions]

        del rootDirContent
        for year, size, session, project in zip( lst_years, lst_sizes, lst_sessions, lst_projects ):
            if type(size) == int: dataYearsContent[year]["Storage Size"] += size
            dataYearsContent[year]["Number of Files"] += 1
            dataYearsContent[year]["Sessions"].add( session )
            dataYearsContent[year]["Projects"].add( project )




print( "Done Loading")        


Loading xc_frag10_yearContent.pkl
10000976
Loading xc_frag11_yearContent.pkl
10000000
Loading xc_frag12_yearContent.pkl
10000843
Loading xc_frag13_yearContent.pkl
10000836
Loading xc_frag14_yearContent.pkl
9294521
Loading xc_frag1_yearContent.pkl
1000
Loading xc_frag2_yearContent.pkl
10000877
Loading xc_frag3_yearContent.pkl
10000000
Loading xc_frag4_yearContent.pkl
10000864
Loading xc_frag5_yearContent.pkl
10000983
Loading xc_frag6_yearContent.pkl
10000792
Loading xc_frag7_yearContent.pkl
10000747
Loading xc_frag8_yearContent.pkl
10000815
Loading xc_frag9_yearContent.pkl
10000000
Done Loading


In [25]:
totalSeshs = 0
totalProjs = 0
for year in sorted( dataYearsContent.keys() ):
    numbSessions = len(dataYearsContent[year]['Sessions'])
    numbProjs = len(dataYearsContent[year]['Projects'])

    print(f"""In {year}: {roundBytes( dataYearsContent[year]['Storage Size'] )} bytes, 
          {dataYearsContent[year]['Number of Files']} files,
          {numbSessions} sessions,
          {numbProjs} projects""")
        
    totalSeshs += numbSessions
    totalProjs += numbProjs

print( f"\nTotal Number of Sessions: {totalSeshs}\nTotal Number of Projects: {totalProjs}")



In 2015: (10.22, 'TB') bytes, 
          24732838 files,
          18 sessions,
          8 projects
In 2016: (28.2, 'TB') bytes, 
          32432270 files,
          72 sessions,
          14 projects
In 2017: (26.76, 'TB') bytes, 
          25736726 files,
          159 sessions,
          34 projects
In 2018: (18.45, 'TB') bytes, 
          15224764 files,
          135 sessions,
          33 projects
In 2019: (22.05, 'TB') bytes, 
          6401641 files,
          111 sessions,
          32 projects
In 2020: (23.88, 'TB') bytes, 
          12278819 files,
          122 sessions,
          46 projects
In 2021: (20.85, 'TB') bytes, 
          10561714 files,
          83 sessions,
          27 projects
In 2022: (1.01, 'TB') bytes, 
          416687 files,
          9 sessions,
          5 projects
In Unknown: (4.28, 'TB') bytes, 
          1517795 files,
          30 sessions,
          14 projects

Total Number of Sessions: 739
Total Number of Projects: 213


In [27]:
# Check why the total numbe of projects when adding between different years is higher
# than when looking at xchem : data/ as a whole.
# Hypothesis: Overlap between Projects that extend for more than a year
lstProjects = [  session[: re.search( "-[0-9]+$", session).start() ] for year in dataYearsContent.keys() for session in dataYearsContent[year]["Sessions"] ]
print( len( set( lstProjects )))


154


In [26]:
yearSizes = [ dataYearsContent[year]["Storage Size"] for year in dataYearsContent.keys() ]
yearfileCount = [ dataYearsContent[year]["Number of Files"] for year in dataYearsContent.keys() ]

yearSeshCount = { year: len( dataYearsContent[year]["Sessions"] ) for year in dataYearsContent.keys() }
yearProjCount = { year: len( dataYearsContent[year]["Projects"] ) for year in dataYearsContent.keys() }




print( "Done Loading")        

meanYearSize = roundBytes( np.mean( yearSizes ) )
medianYearSize = roundBytes( np.median(yearSizes ) )
YearinterQRange = roundBytes( np.percentile( yearSizes, 75) - np.percentile( yearSizes, 25) )

meanYearCount = np.mean( yearfileCount )
medianYearCount = np.median( yearfileCount )
YearinterQRangeCount = np.percentile( yearfileCount, 75) - np.percentile( yearfileCount, 25)

print( "For xchem : data/ per session, we have:")
print( f"\tMean Year Size: {meanYearSize }")
print( f"\tMedian Year Size: {medianYearSize }")
print( f"\tInter Quartile Range of Year Size: {YearinterQRange }")
print( f"\tMean Number of Files per Year: {meanYearCount:.2f}")
print( f"\tMedian Number of Files per Year: {medianYearCount:.2f}")
print( f"\tInter Quartile Range of Number of Files per Year: {YearinterQRangeCount:.2f}")
print( "Done")

Done Loading
For xchem : data/ per session, we have:
	Mean Year Size: (np.float64(17.3), 'TB')
	Median Year Size: (np.float64(20.85), 'TB')
	Inter Quartile Range of Year Size: (np.float64(13.66), 'TB')
	Mean Number of Files per Year: 14367028.22
	Median Number of Files per Year: 12278819.00
	Inter Quartile Range of Number of Files per Year: 18331197.00
Done


### xchem:  dataset/

In [29]:
# xchem : data total projects and sessions
import re
from xaidar.filesUtils import loadPickle

st_sessions = set()
st_projects = set()

xchemSeshDir = Path("./xchemSeshs")
for fragFilePath in xchemSeshDir.iterdir():
    if fragFilePath.is_file():
        # print( f"Loading {fragFilePath.name}")
        rootDirContent = loadPickle( fragFilePath)

        lst_sessions = rootDirContent["dataset"]["session"]
        st_sessions.update( lst_sessions )

        # lst_projects = [ session[:re.search( "-[0-9]+$", session).start() ] for session in lst_sessions]
        # st_projects.update( lst_projects)


print("\nFor xchem : dataset/ there are:")
# print( f"\tNumber of Projects: {len(st_projects )}")
print( f"\tNumber of Sessions: {len(st_sessions) }")


For xchem : dataset/ there are:
	Number of Sessions: 164


In [30]:
# xchem : data/ Per session Data
import numpy as np
from xaidar.filesUtils import roundBytes

dataSessionsContent = { session : {"Storage Size": 0, "Number of Files": 0} for session in st_sessions }


xchemSeshDir = Path("./xchemSeshs")
for fragFilePath in xchemSeshDir.iterdir():
    if fragFilePath.is_file():
        # print( f"Loading {fragFilePath.name}")
        rootDirContent = loadPickle(fragFilePath)
        lst_sessions = rootDirContent["dataset"]["session"] 
        lst_sizes = rootDirContent["dataset"]["size"] 
        del rootDirContent
        for session, size in zip( lst_sessions, lst_sizes ):
            if type(size) == int: dataSessionsContent[session]["Storage Size"] += size
            dataSessionsContent[session]["Number of Files"] += 1

seshSizes = [ dataSessionsContent[sesh]["Storage Size"] for sesh in dataSessionsContent.keys() ]
seshfileCount = [ dataSessionsContent[sesh]["Number of Files"] for sesh in dataSessionsContent.keys() ]

print( "Done Loading")        

meanSeshSize = roundBytes( np.mean( seshSizes ) )
medianSeshSize = roundBytes( np.median(seshSizes ) )
interQRange = roundBytes( np.percentile( seshSizes, 75) - np.percentile( seshSizes, 25) )

meanSeshCount = np.mean( seshfileCount )
medianSeshCount = np.median( seshfileCount )
interQRangeCount = np.percentile( seshfileCount, 75) - np.percentile( seshfileCount, 25)

print( "For xchem : data/ per session, we have:")
print( f"\tMean Session Size: {meanSeshSize }")
print( f"\tMedian Session Size: {medianSeshSize }")
print( f"\tInter Quartile Range of Session Size: {interQRange }")
print( f"\tMean Number of Files per Session: {meanSeshCount:.2f}")
print( f"\tMedian Number of Files per Session: {medianSeshCount:.2f}")
print( f"\tInter Quartile Range of Number of Files per Session: {interQRangeCount:.2f}")
print( "Done")

Done Loading
For xchem : data/ per session, we have:
	Mean Session Size: (np.float64(2.7), 'GB')
	Median Session Size: (np.float64(1.61), 'GB')
	Inter Quartile Range of Session Size: (np.float64(2.6), 'GB')
	Mean Number of Files per Session: 2199.10
	Median Number of Files per Session: 1556.00
	Inter Quartile Range of Number of Files per Session: 2228.75
Done


### pandda

In [44]:
import re
test1 = "fsad.dsaf"
print( re.search( r"\w+\..", test1).group()[:-2] )

fsad


In [45]:
from xaidar.filesUtils import loadPickle

panddaRawKeysDir = Path("../../../data/s3Sizes/pandda/raw")

# saveDir = Path("./panddaSeshs")
# saveDir.mkdir( parents = True, exist_ok = True )


panddaDirContent =  { "session": [], "size": [] } 
for fragPath in panddaRawKeysDir.iterdir(): 
    if fragPath.is_file():
        frag = loadPickle( fragPath )
        for key, size in frag.items():
            objPath = key.split("/")

            if re.search( r"\w+\..", objPath[0] ): 
                panddaDirContent["session"].append(  re.search( r"\w+\..", objPath[0] ).group()[:-2]  ) 
            else:
                panddaDirContent["session"].append( objPath[0] ) 
            
            if type( size ) == int:
                panddaDirContent["size"].append( size )
            else:
                panddaDirContent["size"].append( None)


print( "Done")


Done


In [50]:
st_Sessions = set(  panddaDirContent["session"] )
print( f"In pandda bucket there are {len(st_Sessions)} Sessions")
print(sorted(st_Sessions))

In pandda bucket there are 103 Sessions
['70X', 'AAVNAR', 'AA_VNAR_XF01', 'ACVR1A', 'ATAD2A', 'B2m', 'BKVP1', 'BKVP126', 'BRD1', 'CD44MMA', 'CD73', 'CD73mp', 'CYCK', 'DAPD', 'DCLRE1AA', 'DCP2B', 'EBNA_A2', 'EBNA_T2', 'ERAP1', 'FALZA', 'FabF_C164Q', 'G13D', 'G7', 'GALE', 'GN6S', 'GluN1N2A', 'HAO1A', 'HARBD', 'HEC1', 'HPrP', 'HSP90', 'IAV', 'IDH1', 'JMJD1BA', 'JMJD2AA', 'JMJD2DA', 'JOELS', 'KAT2B', 'KLHL7B', 'KdsA', 'LC_TbrB1', 'LchARH3', 'LmARS', 'M2', 'MACROD1A', 'MID2A', 'MUREECA', 'Mpro', 'NS3Hel', 'NSP15', 'NSP15_B', 'NSP16', 'NUDT21A', 'NUDT22A', 'NUDT4A', 'NUDT5A', 'NUDT7A', 'Nprot', 'OTUB2A', 'OXA10OTA', 'PARG', 'PDE5', 'PDK2', 'PGN_RS02895PGA', 'PHD2', 'PHIPA', 'PIF1', 'PKL1', 'PKM1', 'PP1', 'PRPB', 'PTP1B', 'PWWP_C64S', 'PaFEN_D4', 'PaPBP3', 'PlProwt', 'RECQL5', 'RECQL5A', 'RabC01', 'SETDB1', 'SHH', 'SHMT2A', 'SOCS2A', 'STAG1A', 'TBRUFPPS', 'TBXTA', 'TCRUFPPS', 'TMEMAB', 'TNCA', 'TRF1', 'TRIM2A', 'TbMDO', 'TbrB1', 'TcHRS', 'VP30ctd', 'XX02KALRNA', 'XX11RECQL5A', 'a7ACHBPm', 'mA

In [ ]:
panddaSessionsContent = { session : {"Storage Size": 0, "Number of Files": 0} for session in st_Sessions }

lst_sessions = panddaDirContent["session"] 
lst_sizes = panddaDirContent["size"] 

for session, size in zip( lst_sessions, lst_sizes ):
    if type(size) == int: panddaSessionsContent[session]["Storage Size"] += size
    panddaSessionsContent[session]["Number of Files"] += 1


seshSizes = [ panddaSessionsContent[sesh]["Storage Size"] for sesh in panddaSessionsContent.keys() ]
seshfileCount = [ panddaSessionsContent[sesh]["Number of Files"] for sesh in panddaSessionsContent.keys() ]

meanSeshSize = roundBytes( np.mean( seshSizes ) )
medianSeshSize = roundBytes( np.median(seshSizes ) )
interQRange = roundBytes( np.percentile( seshSizes, 75) - np.percentile( seshSizes, 25) )

meanSeshCount = np.mean( seshfileCount )
medianSeshCount = np.median( seshfileCount )
interQRangeCount = np.percentile( seshfileCount, 75) - np.percentile( seshfileCount, 25)

print( "For pandda per session, we have:")
print( f"\tMean Session Size: {meanSeshSize }")
print( f"\tMedian Session Size: {medianSeshSize }")
print( f"\tInter Quartile Range of Session Size: {interQRange }")
print( f"\tMean Number of Files per Session: {meanSeshCount:.2f}")
print( f"\tMedian Number of Files per Session: {medianSeshCount:.2f}")
print( f"\tInter Quartile Range of Number of Files per Session: {interQRangeCount:.2f}")
print( "Done")

For xchem : data/ per session, we have:
	Mean Session Size: (np.float64(82.28), 'GB')
	Median Session Size: (np.float64(18.13), 'GB')
	Inter Quartile Range of Session Size: (np.float64(46.87), 'GB')
	Mean Number of Files per Session: 49666.62
	Median Number of Files per Session: 24116.00
	Inter Quartile Range of Number of Files per Session: 64168.00
Done
